In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df=pd.read_csv("A:/Projects/Real world data/SaaS Customer Churn/Data/train.csv")

In [9]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

### **Exploratory Data Analysis**

In [39]:
print("First 5 rows:")
print(df.head())

First 5 rows:
                            Customer_ID               Name  \
0  a8b2ac22-dd31-483e-8672-592b7089138f       Brenda Grant   
1  320c00d8-9b88-44f6-ab57-d46976740ca1  Christopher Wells   
2  0c6e183c-4af4-4062-b0aa-875f47a0e0d1     Robert Raymond   
3  c4cf3a80-cbbe-47c5-8340-ef8d149607a7       Glenda Irwin   
4  366001d6-9bec-4667-b80c-18045a682f7f        Alyssa Luna   

                      Email  Account_Age_Days Login_Frequency  \
0       dhanson@example.com               790          Weekly   
1        awatts@example.net               399           Daily   
2   medinaerika@example.org               646           Daily   
3      aramirez@example.org               113          Weekly   
4  bergeramanda@example.org               322          Weekly   

   Daily_Usage_Mins                                     Last_Support_Ticket  \
0                28               Just checking if my payment went through.   
1                28               Just checking if my payment we

In [38]:
print("Dataset Shape:", df.shape)
print("\nColumn Names and Types:")
print(df.dtypes)

Dataset Shape: (2000, 8)

Column Names and Types:
Customer_ID            object
Name                   object
Email                  object
Account_Age_Days        int64
Login_Frequency        object
Daily_Usage_Mins        int64
Last_Support_Ticket    object
Churn                   int64
dtype: object


In [20]:
df.columns

Index(['Customer_ID', 'Name', 'Email', 'Account_Age_Days', 'Login_Frequency',
       'Daily_Usage_Mins', 'Last_Support_Ticket', 'Churn'],
      dtype='object')

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Customer_ID          2000 non-null   object
 1   Name                 2000 non-null   object
 2   Email                2000 non-null   object
 3   Account_Age_Days     2000 non-null   int64 
 4   Login_Frequency      2000 non-null   object
 5   Daily_Usage_Mins     2000 non-null   int64 
 6   Last_Support_Ticket  2000 non-null   object
 7   Churn                2000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 125.1+ KB


#### Checking for null values

In [35]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
Customer_ID            0
Name                   0
Email                  0
Account_Age_Days       0
Login_Frequency        0
Daily_Usage_Mins       0
Last_Support_Ticket    0
Churn                  0
dtype: int64


### Basic Statistics

In [34]:
print("Basic Statistics:")
print(df.describe())

Basic Statistics:
       Account_Age_Days  Daily_Usage_Mins        Churn
count       2000.000000         2000.0000  2000.000000
mean         560.969500           36.1610     0.368500
std          317.349495           29.1486     0.482519
min            1.000000            1.0000     0.000000
25%          287.750000           11.0000     0.000000
50%          565.000000           30.0000     0.000000
75%          840.000000           53.0000     1.000000
max         1094.000000          119.0000     1.000000


### Churn distribution

In [42]:
print("Churn Distribution:")
df['Churn'].value_counts()

Churn Distribution:


Churn
0    1263
1     737
Name: count, dtype: int64

In [47]:
churn_rate=df['Churn'].sum()*100/len(df)
print(f"Churn Rate : {churn_rate}%")

Churn Rate : 36.85%


#### **Key Takeaways:**

-Churn Rate: 36.85% → ~737 customers churned. This is high (SaaS industry avg ~5-7% monthly). Good business problem.

-Account Age: Ranges 1-1,094 days (new to 3-year customers)

-Daily Usage: Ranges 1-119 mins per day (wide spread—some power users, some ghosts)

-Login_Frequency: Categorical (Daily, Weekly, Rarely)

#### Segment Analysis

In [121]:
#Churn by Login Frequency
print("Churn by Login Frequency:")
churn_by_in= df.groupby('Login_Frequency')['Churn'].value_counts().unstack()
print(churn_by_in)
print()

#Churn rate by Login Frequency
print("\nChurn Rate (%) by Login Frequency:")
churn_rate_by_login=df.groupby('Login_Frequency')['Churn'].mean()*100
print(churn_rate_by_login.round(2))

Churn by Login Frequency:
Churn              0    1
Login_Frequency          
Daily            755  113
Rarely            49  288
Weekly           459  336


Churn Rate (%) by Login Frequency:
Login_Frequency
Daily     13.02
Rarely    85.46
Weekly    42.26
Name: Churn, dtype: float64


### **Insight: "Customers who log in rarely are 6.5x more likely to churn than daily users."**

In [110]:
#Create Usage Buckets
df['Usage_Bucket'] = pd.cut(df['Daily_Usage_Mins'], 
                            bins=[0, 15, 50, 120], 
                            labels=['Low (1-15)', 'Medium (15-50)', 'High (50+)'])

# Churn Rate by Usage Buckets
print("Churn Rate (%) by Usage Bucket:")
churn_by_usage = df.groupby('Usage_Bucket',observed=True)['Churn'].mean() * 100
print(churn_by_usage.round(2))

# Also get counts
print("\nChurn Counts by Usage Bucket:")
print(df.groupby('Usage_Bucket',observed=True)['Churn'].value_counts().unstack(fill_value=0))

Churn Rate (%) by Usage Bucket:
Usage_Bucket
Low (1-15)        81.49
Medium (15-50)    17.54
High (50+)        10.29
Name: Churn, dtype: float64

Churn Counts by Usage Bucket:
Churn             0    1
Usage_Bucket            
Low (1-15)      124  546
Medium (15-50)  616  131
High (50+)      523   60


#### **Insight: "Customers using the product <15 mins/day churn at 81%. Those using 50+ mins/day churn at only 10%."**

In [108]:
#Account Age Segments
df['Account_Tenure']= pd.cut(df['Account_Age_Days'], 
                             bins=[0,90,180,365,1100], 
                             labels=['New(0-90d)','Growing(90-180d)','Established(180-365d)','Veteran(365d+)'])

#Churn by Account Tenure
churn_by_tenure=df.groupby('Account_Tenure',observed=all)['Churn'].mean()*100
print(churn_by_tenure.round(2))

#To show the non-churned data also use crosstab

print("\nChurn by Account Tenure:")
churn_by_tenure = pd.crosstab(df['Account_Tenure'], df['Churn'], normalize='index') * 100
print(churn_by_tenure.round(2))

Account_Tenure
New(0-90d)               36.63
Growing(90-180d)         35.42
Established(180-365d)    41.30
Veteran(365d+)           35.98
Name: Churn, dtype: float64

Churn by Account Tenure:
Churn                      0      1
Account_Tenure                     
New(0-90d)             63.37  36.63
Growing(90-180d)       64.58  35.42
Established(180-365d)  58.70  41.30
Veteran(365d+)         64.02  35.98


#### **crosstab() creates a cross-tabulation (contingency table) that shows the frequency/count of two categorical variables together.**

#### Churn is flat across tenure (~35-41%) except 6-12 month accounts churn MORE (41.30%).

Why? (Hypothesis to investigate)

Onboarding effect worn off?
Trial period ended?
Evaluating alternatives after 6 months?
Worth digging into with support tickets

In [120]:
# 4. Average usage for churned vs. retained
print("\n\nAVERAGE METRICS: Churned vs. Retained")
avg_metrics=df.groupby('Churn')[['Account_Age_Days', 'Daily_Usage_Mins']].mean()
print(avg_metrics.round(2))



AVERAGE METRICS: Churned vs. Retained
       Account_Age_Days  Daily_Usage_Mins
Churn                                    
0                568.49             47.68
1                548.08             16.42


#### **Story: Age doesn't matter much. Usage is everything. Retained customers use 3x more than churners.**

## **Statistical Tests**

In [123]:
from scipy.stats import chi2_contingency, ttest_ind

print("="*60)
print("STATISTICAL SIGNIFICANCE TESTS")
print("="*60)

# ===== TEST 1: Login Frequency vs. Churn (Chi-Square) =====
print("\n1. CHI-SQUARE TEST: Login Frequency vs. Churn")
print("-" * 60)

# Create contingency table
contingency_login = pd.crosstab(df['Login_Frequency'], df['Churn'])
print("Contingency Table:")
print(contingency_login)

# Perform chi-square test
chi2_login, p_login, dof_login, expected_login = chi2_contingency(contingency_login)
print(f"\nChi-Square Statistic: {chi2_login:.4f}")
print(f"P-Value: {p_login:.6f}")
print(f"Degrees of Freedom: {dof_login}")

if p_login < 0.05:
    print("✅ RESULT: SIGNIFICANT (p < 0.05)")
    print("   → Login Frequency has a statistically significant relationship with Churn")
else:
    print("❌ RESULT: NOT SIGNIFICANT (p >= 0.05)")

# ===== TEST 2: Usage Bucket vs. Churn (Chi-Square) =====
print("\n\n2. CHI-SQUARE TEST: Usage Bucket vs. Churn")
print("-" * 60)

contingency_usage = pd.crosstab(df['Usage_Bucket'], df['Churn'])
print("Contingency Table:")
print(contingency_usage)

chi2_usage, p_usage, dof_usage, expected_usage = chi2_contingency(contingency_usage)
print(f"\nChi-Square Statistic: {chi2_usage:.4f}")
print(f"P-Value: {p_usage:.6f}")
print(f"Degrees of Freedom: {dof_usage}")

if p_usage < 0.05:
    print("✅ RESULT: SIGNIFICANT (p < 0.05)")
    print("   → Daily Usage Bucket has a statistically significant relationship with Churn")
else:
    print("❌ RESULT: NOT SIGNIFICANT (p >= 0.05)")

# ===== TEST 3: Daily Usage Mins (Churned vs. Retained) - T-Test =====
print("\n\n3. T-TEST: Daily Usage Mins (Churned vs. Retained)")
print("-" * 60)

churned = df[df['Churn'] == 1]['Daily_Usage_Mins']
retained = df[df['Churn'] == 0]['Daily_Usage_Mins']

print(f"Churned Customers - Mean Usage: {churned.mean():.2f} mins (N={len(churned)})")
print(f"Retained Customers - Mean Usage: {retained.mean():.2f} mins (N={len(retained)})")
print(f"Difference: {retained.mean() - churned.mean():.2f} mins")

# Perform independent t-test
t_stat, p_usage_ttest = ttest_ind(churned, retained)
print(f"\nT-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_usage_ttest:.6f}")

if p_usage_ttest < 0.05:
    print("✅ RESULT: SIGNIFICANT (p < 0.05)")
    print("   → Churned and Retained customers have significantly different usage patterns")
else:
    print("❌ RESULT: NOT SIGNIFICANT (p >= 0.05)")

# ===== TEST 4: Account Age (Churned vs. Retained) - T-Test =====
print("\n\n4. T-TEST: Account Age Days (Churned vs. Retained)")
print("-" * 60)

churned_age = df[df['Churn'] == 1]['Account_Age_Days']
retained_age = df[df['Churn'] == 0]['Account_Age_Days']

print(f"Churned Customers - Mean Age: {churned_age.mean():.2f} days (N={len(churned_age)})")
print(f"Retained Customers - Mean Age: {retained_age.mean():.2f} days (N={len(retained_age)})")
print(f"Difference: {retained_age.mean() - churned_age.mean():.2f} days")

t_stat_age, p_age = ttest_ind(churned_age, retained_age)
print(f"\nT-Statistic: {t_stat_age:.4f}")
print(f"P-Value: {p_age:.6f}")

if p_age < 0.05:
    print("✅ RESULT: SIGNIFICANT (p < 0.05)")
    print("   → Churned and Retained customers have significantly different account ages")
else:
    print("❌ RESULT: NOT SIGNIFICANT (p >= 0.05)")
    print("   → Account age is NOT a strong predictor of churn")

# ===== SUMMARY =====
print("\n\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Login Frequency → Churn: p={p_login:.6f} {'✅ SIGNIFICANT' if p_login < 0.05 else '❌ NOT SIGNIFICANT'}")
print(f"Usage Bucket → Churn: p={p_usage:.6f} {'✅ SIGNIFICANT' if p_usage < 0.05 else '❌ NOT SIGNIFICANT'}")
print(f"Daily Usage (t-test): p={p_usage_ttest:.6f} {'✅ SIGNIFICANT' if p_usage_ttest < 0.05 else '❌ NOT SIGNIFICANT'}")
print(f"Account Age (t-test): p={p_age:.6f} {'✅ SIGNIFICANT' if p_age < 0.05 else '❌ NOT SIGNIFICANT'}")

STATISTICAL SIGNIFICANCE TESTS

1. CHI-SQUARE TEST: Login Frequency vs. Churn
------------------------------------------------------------
Contingency Table:
Churn              0    1
Login_Frequency          
Daily            755  113
Rarely            49  288
Weekly           459  336

Chi-Square Statistic: 564.0487
P-Value: 0.000000
Degrees of Freedom: 2
✅ RESULT: SIGNIFICANT (p < 0.05)
   → Login Frequency has a statistically significant relationship with Churn


2. CHI-SQUARE TEST: Usage Bucket vs. Churn
------------------------------------------------------------
Contingency Table:
Churn             0    1
Usage_Bucket            
Low (1-15)      124  546
Medium (15-50)  616  131
High (50+)      523   60

Chi-Square Statistic: 870.2458
P-Value: 0.000000
Degrees of Freedom: 2
✅ RESULT: SIGNIFICANT (p < 0.05)
   → Daily Usage Bucket has a statistically significant relationship with Churn


3. T-TEST: Daily Usage Mins (Churned vs. Retained)
------------------------------------------

In [124]:
# Export the dataframe with all calculated columns
df.to_csv('saas_churn_analysis.csv', index=False)
print("File exported: saas_churn_analysis.csv")

File exported: saas_churn_analysis.csv


C:\Users\Adnan Ayoub Dar\Projects
